In [1]:
import os, random, gc, itertools, json
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

In [2]:
NEW_DATASET_DIR = "/kaggle/input/datasets/almiraraisa/aptos-2019-rev" 
NEW_TRAIN_CSV   = f"{NEW_DATASET_DIR}/train_split.csv"
NEW_VAL_CSV     = f"{NEW_DATASET_DIR}/val_split.csv"
NEW_TEST_CSV    = f"{NEW_DATASET_DIR}/test_split.csv"
CKPT_DIR        = f"/kaggle/working"

DATASET_DIR = NEW_DATASET_DIR
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES   = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
NUM_CLASSES   = len(CLASS_NAMES)

train_df = pd.read_csv(NEW_TRAIN_CSV)
val_df   = pd.read_csv(NEW_VAL_CSV)
test_df  = pd.read_csv(NEW_TEST_CSV)

NUM_EPOCHS    = 100
SEARCH_SEED   = 33
SEARCH_EPOCHS = 40

class_counts   = train_df['diagnosis'].value_counts().sort_index()
class_weights  = 1.0 / class_counts
sample_weights = train_df['diagnosis'].map(class_weights).values

print(f"  [dataset] train={len(train_df)}  val={len(val_df)}  test={len(test_df)}  "
      f"from {NEW_DATASET_DIR}")

  [dataset] train=20503  val=366  test=367  from /kaggle/input/datasets/almiraraisa/aptos-2019-rev


In [3]:
def save_phase_results(phase_name, results):
    path = f'{CKPT_DIR}/phase_{phase_name}.json'
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  [checkpoint] Phase '{phase_name}' results saved → {path}")

def load_phase_results(phase_name):
    path = f'{CKPT_DIR}/phase_{phase_name}.json'
    if os.path.exists(path):
        with open(path, 'r') as f:
            results = json.load(f)
        print(f"  [checkpoint] Phase '{phase_name}' already done — loaded from {path}")
        return results
    return None

def phase_is_done(phase_name):
    return os.path.exists(f'{CKPT_DIR}/phase_{phase_name}.json')

In [4]:
def save_best_weights(path, epoch, model, qwk):
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'qwk': qwk}, path)

def save_resume_ckpt(path, epoch, model, optimizer, scheduler, scaler,
                     history, best_qwk, best_preds, best_labels):
    torch.save({'epoch':            epoch,
                'model_state_dict': model.state_dict(),
                'optimizer':        optimizer.state_dict(),
                'scheduler':        scheduler.state_dict(),
                'scaler':           scaler.state_dict(),
                'history':          history,
                'best_qwk':         best_qwk,
                'best_preds':       best_preds,
                'best_labels':      best_labels}, path)

def load_resume_ckpt(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])

    scaler_state = ckpt.get('scaler')
    if scaler_state:  # non-empty dict -> safe to load
        scaler.load_state_dict(scaler_state)
    else:
        print("  [warning] checkpoint's GradScaler state was empty "
              "(saved from a disabled/CPU-mode scaler) — "
              "continuing with a freshly-initialized scaler instead")

    print(f"  [checkpoint] Resumed from epoch {ckpt['epoch']+1} "
          f"(best QWK so far: {ckpt['best_qwk']:.4f})")
    return (ckpt['epoch'] + 1, ckpt['history'],
            ckpt['best_qwk'], ckpt['best_preds'], ckpt['best_labels'])

In [5]:
def set_seed(seed=30):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def _compute_mse_rmse(labels, preds):
    la = np.array(labels, dtype=np.float32)
    pa = np.array(preds,  dtype=np.float32)
    mse  = float(np.mean((la - pa) ** 2))
    return mse, float(np.sqrt(mse)), float(np.mean(np.abs(la - pa)))

In [6]:
class AptosDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img_col = 'id_code' if 'id_code' in self.df.columns else self.df.columns[0]
        fname   = str(row[img_col])
        fname   = fname if fname.endswith('.png') else fname + '.png'
        img     = Image.open(self.image_dir / fname).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(int(row['diagnosis']), dtype=torch.long)

base_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_df = pd.read_csv(f'{DATASET_DIR}/train_split.csv')
val_df   = pd.read_csv(f'{DATASET_DIR}/val_split.csv')
test_df  = pd.read_csv(f'{DATASET_DIR}/test_split.csv')
train_df['diagnosis'] = train_df['diagnosis'].astype(int)

class_counts   = train_df['diagnosis'].value_counts().sort_index().values
sample_weights = train_df['diagnosis'].map(lambda x: 1.0 / class_counts[x]).values

val_loader  = DataLoader(AptosDataset(val_df,  f'{DATASET_DIR}/images', base_tfm),
                         batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(AptosDataset(test_df, f'{DATASET_DIR}/images', base_tfm),
                         batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

def make_train_loader(batch_size):
    return DataLoader(
        AptosDataset(train_df, f'{DATASET_DIR}/images', base_tfm),
        batch_size = batch_size,
        sampler    = WeightedRandomSampler(
            torch.tensor(sample_weights, dtype=torch.float),
            len(sample_weights), replacement=True),
        num_workers=4, pin_memory=True)

In [7]:
class CoralHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc   = nn.Linear(in_features, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_classes - 1))
    def forward(self, x):
        return self.fc(x) + self.bias

def coral_loss(logits, labels, num_classes=NUM_CLASSES):
    targets = torch.zeros(labels.size(0), num_classes - 1, device=labels.device)
    for k in range(num_classes - 1):
        targets[:, k] = (labels > k).float()
    return F.binary_cross_entropy_with_logits(logits, targets)

def coral_predict(logits):
    return (torch.sigmoid(logits) > 0.5).sum(dim=1).long()

def coral_to_softmax(logits):
    cum   = torch.sigmoid(logits)
    p0    = 1.0 - cum[:, 0:1]
    pmid  = cum[:, :-1] - cum[:, 1:]
    plast = cum[:, -1:]
    return torch.cat([p0, pmid, plast], dim=1).clamp(min=1e-6)

def coral_kd_loss(s_logits, t_logits, labels, alpha, temperature, beta):
    ord_loss     = coral_loss(s_logits, labels)
    s_probs      = coral_to_softmax(s_logits)
    t_soft       = F.softmax(t_logits / temperature, dim=1)
    s_soft       = s_probs ** (1.0 / temperature)
    s_soft       = s_soft / s_soft.sum(dim=1, keepdim=True)
    s_log_t      = torch.log(s_soft + 1e-8)
    kd_loss      = F.kl_div(s_log_t, t_soft, reduction='batchmean') * (temperature ** 2)
    return (1 - alpha) * beta * ord_loss + alpha * kd_loss

def make_mobilenet_ce(dropout):
    m       = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_feat = m.classifier[-1].in_features
    m.classifier[-1] = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_feat, NUM_CLASSES))
    return m

def make_mobilenet_coral(dropout):
    m       = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    in_feat = m.classifier[-1].in_features
    m.classifier[-1] = nn.Dropout(p=dropout)
    m.coral_head     = CoralHead(in_feat, NUM_CLASSES)
    return m

def get_resnet50_teacher():
    m    = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m

In [8]:
# ----------------------------------------------------------------------
# STEP 0 — student architecture search (LR×WD, then BS×Dropout)
#          Must run BEFORE the loaders below, since they consume best_bs.
# ----------------------------------------------------------------------

# dataset changed -> old arch-search results are not valid for this data
# def clear_stale_phase(*phase_names):
#     for name in phase_names:
#         p = f'{CKPT_DIR}/phase_{name}.json'
#         if os.path.exists(p):
#             os.remove(p)
#             print(f"  [reset] removed stale phase results: {p}")

# clear_stale_phase('stage0a', 'stage0b')

ARCH_SEARCH = {
    'lr'          : [1e-3, 5e-4, 1e-4, 5e-5],
    'weight_decay': [1e-4, 1e-5, 1e-6],
    'batch_size'  : [16, 32, 64],
    'dropout'     : [0.2, 0.3, 0.4, 0.5],
}

DISTILL_SEARCH = {
    'temperature': [3, 5, 7, 10],
    'alpha'      : [0.3, 0.5, 0.7, 0.9],
}
BETA_SEARCH = [0.5, 1.0, 2.0]

ARCH_SEARCH_EPOCHS = 20   # vs SEARCH_EPOCHS=40; halves Stage-0 wall-clock cost

def make_train_loader_plain(batch_size):
    """Temporary loader used only during arch search — not the shared
    train_loader (that gets built once, after best_bs is known)."""
    return DataLoader(
        AptosDataset(train_df, f'{DATASET_DIR}/images', base_tfm),
        batch_size=batch_size,
        sampler=WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.float),
                                       len(sample_weights), replacement=True),
        num_workers=4, pin_memory=True)

def train_ce_only_arch_search(lr, weight_decay, batch_size, dropout,
                               num_epochs=ARCH_SEARCH_EPOCHS):
    loader = make_train_loader_plain(batch_size)
    set_seed(SEARCH_SEED)
    model  = make_mobilenet_ce(dropout).to(device)
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler = torch.amp.GradScaler('cuda')
    best_qwk = -1.0
    for epoch in range(num_epochs):
        model.train()
        for imgs, labels in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = F.cross_entropy(model(imgs), labels)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        model.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                v_preds.extend(model(imgs.to(device)).argmax(dim=1).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        if qwk > best_qwk: best_qwk = qwk
    del model; cleanup_memory()
    return best_qwk

# ---- STAGE 0A — LR × Weight Decay (BS=32, Dropout=0.3 fixed) ----
stage0a_results = load_phase_results('stage0a')
if stage0a_results is None:
    print("\n" + "="*65)
    print("  STAGE 0A — LR × Weight Decay  (BS=32, Dropout=0.3 fixed)")
    print("="*65)
    stage0a_results = []
    for lr, wd in itertools.product(ARCH_SEARCH['lr'], ARCH_SEARCH['weight_decay']):
        print(f"  Trial | LR={lr}  WD={wd}", end='  ', flush=True)
        try:
            qwk = train_ce_only_arch_search(lr=lr, weight_decay=wd, batch_size=32, dropout=0.3)
        except Exception as e:
            save_phase_results('stage0a', stage0a_results)
            print(f"\n  [checkpoint] saved {len(stage0a_results)} trials before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        stage0a_results.append({'lr': lr, 'wd': wd, 'qwk': qwk})
        save_phase_results('stage0a', stage0a_results)   # checkpoint after EVERY trial
    stage0a_results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results('stage0a', stage0a_results)

best_lr = stage0a_results[0]['lr']
best_wd = stage0a_results[0]['wd']
print(f"\n  Stage 0A best: LR={best_lr}  WD={best_wd}  QWK={stage0a_results[0]['qwk']:.4f}")

# ---- STAGE 0B — Batch Size × Dropout (best_lr, best_wd fixed) ----
stage0b_results = load_phase_results('stage0b')
if stage0b_results is None:
    print("\n" + "="*65)
    print(f"  STAGE 0B — Batch Size × Dropout  (LR={best_lr}, WD={best_wd} fixed)")
    print("="*65)
    stage0b_results = []
    for bs, dr in itertools.product(ARCH_SEARCH['batch_size'], ARCH_SEARCH['dropout']):
        print(f"  Trial | BS={bs}  Dropout={dr}", end='  ', flush=True)
        try:
            qwk = train_ce_only_arch_search(lr=best_lr, weight_decay=best_wd, batch_size=bs, dropout=dr)
        except Exception as e:
            save_phase_results('stage0b', stage0b_results)
            print(f"\n  [checkpoint] saved {len(stage0b_results)} trials before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        stage0b_results.append({'bs': bs, 'dropout': dr, 'qwk': qwk})
        save_phase_results('stage0b', stage0b_results)
    stage0b_results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results('stage0b', stage0b_results)

best_bs      = stage0b_results[0]['bs']
best_dropout = stage0b_results[0]['dropout']
print(f"\n  Stage 0B best: BS={best_bs}  Dropout={best_dropout}  QWK={stage0b_results[0]['qwk']:.4f}")

print("\n" + "="*65)
print("  STAGE 0 COMPLETE — Shared Architecture Config")
print(f"  LR={best_lr}  WD={best_wd}  BS={best_bs}  Dropout={best_dropout}")
print("="*65)

  [checkpoint] Phase 'stage0a' already done — loaded from /kaggle/working/phase_stage0a.json

  Stage 0A best: LR=0.001  WD=1e-05  QWK=0.9005
  [checkpoint] Phase 'stage0b' already done — loaded from /kaggle/working/phase_stage0b.json

  Stage 0B best: BS=16  Dropout=0.2  QWK=0.8770

  STAGE 0 COMPLETE — Shared Architecture Config
  LR=0.001  WD=1e-05  BS=16  Dropout=0.2


In [9]:
# ----------------------------------------------------------------------
# STEP 1 — datasets / loaders (plain, for teacher training+eval; and an
#          index-returning variant used later to cache + look up KD logits)
# ----------------------------------------------------------------------
class AptosDataset(torch.utils.data.Dataset):
    """Plain (img, label) dataset — used for teacher train/val/test loaders."""
    def __init__(self, df, image_dir, transform=None):
        self.df, self.image_dir, self.transform = df.reset_index(drop=True), Path(image_dir), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img_col = 'id_code' if 'id_code' in self.df.columns else self.df.columns[0]
        fname   = str(row[img_col]); fname = fname if fname.endswith('.png') else fname + '.png'
        img     = Image.open(self.image_dir / fname).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(int(row['diagnosis']), dtype=torch.long)


class AptosDatasetIdx(torch.utils.data.Dataset):
    """Same as above but also returns row idx, for caching teacher logits."""
    def __init__(self, df, image_dir, transform=None):
        self.df, self.image_dir, self.transform = df.reset_index(drop=True), Path(image_dir), transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        img_col = 'id_code' if 'id_code' in self.df.columns else self.df.columns[0]
        fname   = str(row[img_col]); fname = fname if fname.endswith('.png') else fname + '.png'
        img     = Image.open(self.image_dir / fname).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(int(row['diagnosis']), dtype=torch.long), idx


train_loader = DataLoader(
    AptosDataset(train_df, f'{DATASET_DIR}/images', base_tfm),
    batch_size=best_bs,
    sampler=WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.float),
                                   len(sample_weights), replacement=True),
    num_workers=4, pin_memory=True)

val_loader = DataLoader(AptosDataset(val_df, f'{DATASET_DIR}/images', base_tfm),
                         batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

test_loader = DataLoader(AptosDataset(test_df, f'{DATASET_DIR}/images', base_tfm),
                          batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

cache_loader = DataLoader(AptosDatasetIdx(train_df, f'{DATASET_DIR}/images', base_tfm),
                          batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

def make_train_loader_idx(batch_size):
    return DataLoader(
        AptosDatasetIdx(train_df, f'{DATASET_DIR}/images', base_tfm),
        batch_size=batch_size,
        sampler=WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.float),
                                       len(sample_weights), replacement=True),
        num_workers=4, pin_memory=True)

In [10]:
# ----------------------------------------------------------------------
# STEP 2 — coral_logits_to_probs (needed to unify CE / CORAL teacher spaces)
# ----------------------------------------------------------------------
def coral_logits_to_probs(logits):
    """P(Y>k)=sigmoid(logit_k); p(Y=k)=P(Y>k-1)-P(Y>k). Returns (B,K) simplex."""
    B = logits.size(0)
    cum        = torch.sigmoid(logits)
    ones       = torch.ones (B, 1, device=logits.device)
    zeros      = torch.zeros(B, 1, device=logits.device)
    cum_padded = torch.cat([ones, cum, zeros], dim=1)
    probs      = (cum_padded[:, :-1] - cum_padded[:, 1:]).clamp(min=1e-8)
    return probs / probs.sum(dim=1, keepdim=True)

In [11]:
# ----------------------------------------------------------------------
# STEP 3 — model builders for BOTH teachers (ImageNet-pretrained backbone,
#          freshly initialized head — trained end-to-end from here)
# ----------------------------------------------------------------------
def make_resnet50_ce():
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_feat = m.fc.in_features
    m.fc = nn.Linear(in_feat, NUM_CLASSES)
    return m

def fwd_resnet50_ce(model, imgs):
    return model(imgs)

def make_efficientnet_b3_coral():
    m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_feat = m.classifier[-1].in_features
    m.classifier[-1] = nn.Identity()
    m.coral_head = CoralHead(in_feat, NUM_CLASSES)
    return m

def fwd_efficientnet_coral(model, imgs):
    feat = model.features(imgs)
    feat = model.avgpool(feat).flatten(1)
    return model.coral_head(feat)

In [12]:
# ----------------------------------------------------------------------
# STEP 4 — reset stale artifacts from the previous dataset, then train
#          BOTH teachers from scratch with one shared training routine
# ----------------------------------------------------------------------
# def clear_stale_checkpoints(*save_paths):
#     """Delete best/resume checkpoints so training doesn't silently resume
#     from weights fit to the old dataset."""
#     for save_path in save_paths:
#         resume_path = save_path.replace('.pth', '_resume.pth')
#         for p in (save_path, resume_path):
#             if os.path.exists(p):
#                 os.remove(p)
#                 print(f"  [reset] removed stale checkpoint: {p}")

def train_teacher(model, fwd_fn, loss_fn, predict_fn, name, save_path,
                   seed=SEARCH_SEED, num_epochs=NUM_EPOCHS):
    """Generic teacher trainer: works for the CE teacher (CrossEntropyLoss +
    argmax) and the CORAL teacher (coral_loss + coral_predict) alike."""
    resume_path = save_path.replace('.pth', '_resume.pth')
    set_seed(seed)
    model  = model.to(device)
    opt    = optim.AdamW(model.parameters(), lr=best_lr, weight_decay=best_wd)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler = torch.amp.GradScaler('cuda')
    best_qwk, start_epoch = -1.0, 0
    history = {'train_loss': [], 'val_qwk': []}
    best_preds, best_labels = [], []

    if os.path.exists(resume_path):
        start_epoch, history, best_qwk, best_preds, best_labels = \
            load_resume_ckpt(resume_path, model, opt, sched, scaler)

    for epoch in range(start_epoch, num_epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                loss = loss_fn(fwd_fn(model, imgs), labels)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        model.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                logits = fwd_fn(model, imgs.to(device))
                v_preds.extend(predict_fn(logits))
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        history['val_qwk'].append(qwk)
        print(f"  [{name}] Epoch {epoch+1:03d} | QWK={qwk:.4f}")
        if qwk > best_qwk:
            best_qwk = qwk; best_preds, best_labels = list(v_preds), list(v_labels)
            save_best_weights(save_path, epoch + 1, model, best_qwk)
        save_resume_ckpt(resume_path, epoch, model, opt, sched, scaler,
                         history, best_qwk, best_preds, best_labels)

    model.load_state_dict(torch.load(save_path, map_location=device,
                                     weights_only=False)['model_state_dict'])
    model.eval()
    print(f"  {name} best val QWK: {best_qwk:.4f}")
    return model, best_qwk


teacher_ce_save    = f'{CKPT_DIR}/teacher_resnet50_ce_shared.pth'
teacher_coral_save = f'{CKPT_DIR}/teacher_efficientnetb3_coral_shared.pth'
teacher_cache_path = f'{CKPT_DIR}/teacher_logit_cache.pt'

# dataset changed -> old teacher weights + cached logits are no longer valid
# clear_stale_checkpoints(teacher_ce_save, teacher_coral_save)
if os.path.exists(teacher_cache_path):
    os.remove(teacher_cache_path)
    print(f"  [reset] removed stale teacher logit cache: {teacher_cache_path}")

set_seed(SEARCH_SEED)
shared_teacher, ce_qwk = train_teacher(
    make_resnet50_ce(), fwd_resnet50_ce,
    loss_fn=nn.CrossEntropyLoss(),
    predict_fn=lambda logits: logits.argmax(dim=1).cpu().numpy(),
    name="Teacher-CE(ResNet-50)", save_path=teacher_ce_save)
cleanup_memory()

set_seed(SEARCH_SEED)
teacher_coral, tc_qwk = train_teacher(
    make_efficientnet_b3_coral(), fwd_efficientnet_coral,
    loss_fn=coral_loss,
    predict_fn=lambda logits: coral_predict(logits).cpu().numpy(),
    name="Teacher-CORAL(EffNet-B3)", save_path=teacher_coral_save)
cleanup_memory()

  [reset] removed stale teacher logit cache: /kaggle/working/teacher_logit_cache.pt
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s] 


  [checkpoint] Resumed from epoch 100 (best QWK so far: 0.9015)
  Teacher-CE(ResNet-50) best val QWK: 0.9015
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 166MB/s] 


  [checkpoint] Resumed from epoch 100 (best QWK so far: 0.9158)
  Teacher-CORAL(EffNet-B3) best val QWK: 0.9158


In [13]:
# ----------------------------------------------------------------------
# STEP 5 — cache both teachers' logits ONCE (the key efficiency win)
# ----------------------------------------------------------------------
def build_teacher_cache():
    cache_path = teacher_cache_path
    if os.path.exists(cache_path):
        print("  [cache] Teacher logit cache found — loading from disk")
        d = torch.load(cache_path, map_location='cpu')
        return d['ce_logits'], d['coral_logits']

    print("  [cache] Building teacher logit cache (one-time pass over train set)...")
    N = len(train_df)
    ce_logits    = torch.zeros(N, NUM_CLASSES)
    coral_logits = torch.zeros(N, NUM_CLASSES - 1)
    t_ce    = shared_teacher.to(device).eval()
    t_coral = teacher_coral.to(device).eval()
    with torch.no_grad():
        for imgs, labels, idx in cache_loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                ce_out    = t_ce(imgs)
                coral_out = fwd_efficientnet_coral(t_coral, imgs)
            ce_logits[idx]    = ce_out.float().cpu()
            coral_logits[idx] = coral_out.float().cpu()
    torch.save({'ce_logits': ce_logits, 'coral_logits': coral_logits}, cache_path)
    print(f"  [cache] Saved to {cache_path}")
    return ce_logits, coral_logits

TEACHER_CE_LOGITS, TEACHER_CORAL_LOGITS = build_teacher_cache()
cleanup_memory()

  [cache] Building teacher logit cache (one-time pass over train set)...
  [cache] Saved to /kaggle/working/teacher_logit_cache.pt


In [14]:
T_SPACE = [1.0, 2.0, 4.0, 6.0]


def _simple_avg_kd_loss(s_logits, t_ce_l, t_coral_l, labels, temperature, alpha=0.5):
    p_ce    = F.softmax(t_ce_l / temperature, dim=1)
    p_coral = F.softmax(torch.log(coral_logits_to_probs(t_coral_l).clamp(min=1e-8)) / temperature, dim=1)
    p_avg   = 0.5 * (p_ce + p_coral)
    s_log   = F.log_softmax(s_logits / temperature, dim=1)
    kl      = (p_avg * (p_avg.log() - s_log)).sum(dim=1).mean() * (temperature ** 2)
    return (1 - alpha) * F.cross_entropy(s_logits, labels) + alpha * kl


def train_search_phase1b(temperature, num_epochs=SEARCH_EPOCHS):
    set_seed(SEARCH_SEED)
    loader = make_train_loader_idx(best_bs)
    student = make_mobilenet_ce(best_dropout).to(device)
    opt    = optim.AdamW(student.parameters(), lr=best_lr, weight_decay=best_wd)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler = torch.amp.GradScaler('cuda')
    ce_cache, coral_cache = TEACHER_CE_LOGITS.to(device), TEACHER_CORAL_LOGITS.to(device)
    best_qwk = -1.0

    for epoch in range(num_epochs):
        student.train()
        for imgs, labels, idx in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t_ce_l, t_coral_l = ce_cache[idx], coral_cache[idx]
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                s_logits = student(imgs)
                loss = _simple_avg_kd_loss(s_logits, t_ce_l, t_coral_l, labels, temperature)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        student.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                v_preds.extend(student(imgs.to(device)).argmax(dim=1).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        if qwk > best_qwk: best_qwk = qwk

    del student; cleanup_memory()
    return best_qwk


def hypersearch_phase1b(phase_name='phase1b_temperature'):
    results = load_phase_results(phase_name) or []
    done = {r['temperature'] for r in results}
    remaining = [t for t in T_SPACE if t not in done]
    print(f"\n{'='*65}\n  PHASE 1b — temperature search\n"
          f"  {len(T_SPACE)} trials total, {len(results)} already done, "
          f"{len(remaining)} remaining\n{'='*65}")

    for t in remaining:
        print(f"  Trial | T={t}", end='  ', flush=True)
        try:
            qwk = train_search_phase1b(t)
        except Exception as e:
            save_phase_results(phase_name, results)
            print(f"\n  [checkpoint] saved {len(results)}/{len(T_SPACE)} trials before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        results.append({'temperature': t, 'qwk': qwk})
        save_phase_results(phase_name, results)  # checkpoint after EVERY trial

    results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results(phase_name, results)
    return results


phase1b_results = hypersearch_phase1b()
best_phase1b = phase1b_results[0]
T_fixed = best_phase1b['temperature']

print(f"\n  PHASE 1b best: T={T_fixed}  QWK={best_phase1b['qwk']:.4f}")

  [checkpoint] Phase 'phase1b_temperature' already done — loaded from /kaggle/working/phase_phase1b_temperature.json

  PHASE 1b — temperature search
  4 trials total, 4 already done, 0 remaining
  [checkpoint] Phase 'phase1b_temperature' results saved → /kaggle/working/phase_phase1b_temperature.json

  PHASE 1b best: T=6.0  QWK=0.8938


In [15]:
# ----------------------------------------------------------------------
# STEP 6 — multi-teacher KD losses (unify CE + CORAL teacher spaces)
# ----------------------------------------------------------------------
def mkd_avg_loss(s_logits, t_ce_l, t_coral_l, labels, alpha, temperature, **_):
    """Simple average of two teachers' softened probabilities (nominal space)."""
    p_ce    = F.softmax(t_ce_l / temperature, dim=1)
    p_coral = F.softmax(torch.log(coral_logits_to_probs(t_coral_l).clamp(min=1e-8)) / temperature, dim=1)
    p_avg   = 0.5 * (p_ce + p_coral)
    s_log   = F.log_softmax(s_logits / temperature, dim=1)
    kl      = (p_avg * (p_avg.log() - s_log)).sum(dim=1).mean() * (temperature ** 2)
    return (1 - alpha) * F.cross_entropy(s_logits, labels) + alpha * kl


def camkd_loss(s_logits, t_ce_l, t_coral_l, labels, alpha, temperature, **_):
    """Confidence-Aware Multi-Teacher KD (Zhang et al., ICASSP 2022, arXiv:2201.00007)."""
    p_ce    = F.softmax(t_ce_l / temperature, dim=1)
    p_coral = F.softmax(torch.log(coral_logits_to_probs(t_coral_l).clamp(min=1e-8)) / temperature, dim=1)
    y_oh       = F.one_hot(labels, NUM_CLASSES).float()
    conf_ce    = torch.exp(-(-(y_oh * (p_ce    + 1e-8).log()).sum(dim=1)))
    conf_coral = torch.exp(-(-(y_oh * (p_coral + 1e-8).log()).sum(dim=1)))
    denom      = conf_ce + conf_coral + 1e-8
    w_ce, w_coral = (conf_ce / denom).unsqueeze(1), (conf_coral / denom).unsqueeze(1)
    p_agg   = w_ce * p_ce + w_coral * p_coral
    s_log   = F.log_softmax(s_logits / temperature, dim=1)
    kl      = (p_agg * (p_agg.log() - s_log)).sum(dim=1).mean() * (temperature ** 2)
    return (1 - alpha) * F.cross_entropy(s_logits, labels) + alpha * kl


def mkd_coral_loss(s_coral_logits, t_ce_l, t_coral_l, labels, alpha, temperature, beta=1.0):
    """Ordinal-space multi-teacher KD: student is CORAL; teachers averaged in
    cumulative P(Y>k) space (CE teacher's softmax is folded into cumulative form)."""
    p_ce_softmax = F.softmax(t_ce_l / temperature, dim=1)
    ce_cum       = 1.0 - torch.cumsum(p_ce_softmax, dim=1)[:, :-1]      # P(Y>k) from CE teacher
    coral_cum    = torch.sigmoid(t_coral_l / temperature)               # P(Y>k) from CORAL teacher
    p_avg_cum    = 0.5 * (ce_cum + coral_cum)
    s_cum        = torch.sigmoid(s_coral_logits / temperature)
    kd_loss      = F.mse_loss(s_cum, p_avg_cum)
    hard_loss    = coral_loss(s_coral_logits, labels)
    return (1 - alpha) * beta * hard_loss + alpha * kd_loss


MULTIKD_METHODS = {
    'mkd_avg':   {'loss_fn': mkd_avg_loss,   'coral_student': False},
    'camkd':     {'loss_fn': camkd_loss,     'coral_student': False},
    'mkd_coral': {'loss_fn': mkd_coral_loss, 'coral_student': True},
}

In [16]:
# ----------------------------------------------------------------------
# STEP 7 — fast search trial (uses cached logits, no teacher forward pass)
# ----------------------------------------------------------------------
def train_search_multikd(method, alpha, temperature, beta=1.0, num_epochs=SEARCH_EPOCHS):
    cfg    = MULTIKD_METHODS[method]
    loader = make_train_loader_idx(best_bs)
    set_seed(SEARCH_SEED)
    student = (make_mobilenet_coral(best_dropout) if cfg['coral_student']
               else make_mobilenet_ce(best_dropout)).to(device)
    opt    = optim.AdamW(student.parameters(), lr=best_lr, weight_decay=best_wd)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler = torch.amp.GradScaler('cuda')
    ce_cache, coral_cache = TEACHER_CE_LOGITS.to(device), TEACHER_CORAL_LOGITS.to(device)
    best_qwk = -1.0

    for epoch in range(num_epochs):
        student.train()
        for imgs, labels, idx in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t_ce_l, t_coral_l = ce_cache[idx], coral_cache[idx]   # cheap lookup, no teacher forward
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                if cfg['coral_student']:
                    feat     = student.classifier(student.features(imgs).mean([2, 3]))
                    s_logits = student.coral_head(feat)
                else:
                    s_logits = student(imgs)
                loss = cfg['loss_fn'](s_logits, t_ce_l, t_coral_l, labels,
                                      alpha=alpha, temperature=temperature, beta=beta)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        student.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                if cfg['coral_student']:
                    feat   = student.classifier(student.features(imgs.to(device)).mean([2, 3]))
                    logits = student.coral_head(feat)
                    v_preds.extend(coral_predict(logits).cpu().numpy())
                else:
                    v_preds.extend(student(imgs.to(device)).argmax(dim=1).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        if qwk > best_qwk: best_qwk = qwk

    del student; cleanup_memory()
    return best_qwk

In [17]:
# ----------------------------------------------------------------------
# STEP 8 — hyperparameter search: reuse T_fixed, search only alpha (+beta)
#          NOW WITH PER-TRIAL CHECKPOINTING (resumes mid-phase, not just
#          at phase boundaries)
# ----------------------------------------------------------------------
def hypersearch_multikd(method, alpha_space, beta_space, phase_name, tag):
    trials = list(itertools.product(alpha_space, beta_space))

    # Load whatever partial progress exists for this phase (if any)
    results = load_phase_results(phase_name) or []
    done_keys = {(r['alpha'], r['beta']) for r in results}

    if len(results) == len(trials) and all(
        (a, b) in done_keys for a, b in trials
    ):
        results.sort(key=lambda x: x['qwk'], reverse=True)
        print(f"  {tag} — already complete. Best: α={results[0]['alpha']}  "
              f"β={results[0]['beta']}  QWK={results[0]['qwk']:.4f}")
        return results

    remaining = [(a, b) for a, b in trials if (a, b) not in done_keys]
    print(f"\n{'='*65}\n  {tag}\n  {len(trials)} trials total, "
          f"{len(results)} already done, {len(remaining)} remaining  "
          f"(T={T_fixed}, reused from Phase 1)\n{'='*65}")

    for alpha, beta in remaining:
        print(f"  Trial | α={alpha}  β={beta}", end='  ', flush=True)
        try:
            qwk = train_search_multikd(method, alpha=alpha, temperature=T_fixed, beta=beta)
        except Exception as e:
            # Save what we have before propagating, so a crash here doesn't
            # cost the trials already completed in this call
            save_phase_results(phase_name, results)
            print(f"\n  [checkpoint] saved {len(results)}/{len(trials)} trials "
                  f"for {phase_name} before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        results.append({'alpha': alpha, 'beta': beta, 'qwk': qwk})
        save_phase_results(phase_name, results)   # checkpoint after EVERY trial

    results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results(phase_name, results)
    return results

In [18]:
# ----------------------------------------------------------------------
# STEP 8b — PHASE 2: tune alpha (and beta, for mkd_coral) per multi-KD
#           method, reusing the cached teacher logits from STEP 5/7 —
#           no teacher forward pass here, same efficiency win as Phase 1b.
# ----------------------------------------------------------------------
# clear_stale_phase('phase2a_mkd_avg', 'phase2b_camkd', 'phase2c_mkd_coral')

# mkd_avg and camkd don't use `beta` in their loss formulas (see STEP 6),
# so gridding beta for them would just re-run identical trials — search
# a single beta=1.0 "placeholder" instead. mkd_coral genuinely uses beta
# to weight the ordinal hard-loss term, so it gets the full BETA_SEARCH grid.
mkd_avg_results = hypersearch_multikd(
    method='mkd_avg', alpha_space=DISTILL_SEARCH['alpha'], beta_space=[1.0],
    phase_name='phase2a_mkd_avg', tag=f"PHASE 2a — Tune α for mkd_avg  (T={T_fixed})")
best_mkd_avg = mkd_avg_results[0]

camkd_results = hypersearch_multikd(
    method='camkd', alpha_space=DISTILL_SEARCH['alpha'], beta_space=[1.0],
    phase_name='phase2b_camkd', tag=f"PHASE 2b — Tune α for camkd  (T={T_fixed})")
best_camkd = camkd_results[0]

mkd_coral_results = hypersearch_multikd(
    method='mkd_coral', alpha_space=DISTILL_SEARCH['alpha'], beta_space=BETA_SEARCH,
    phase_name='phase2c_mkd_coral', tag=f"PHASE 2c — Tune α × β for mkd_coral  (T={T_fixed})")
best_mkd_coral = mkd_coral_results[0]

print("\n" + "="*65)
print("  PHASE 2 COMPLETE")
print("="*65)
print(f"  mkd_avg:   α={best_mkd_avg['alpha']}                      → QWK={best_mkd_avg['qwk']:.4f}")
print(f"  camkd:     α={best_camkd['alpha']}                        → QWK={best_camkd['qwk']:.4f}")
print(f"  mkd_coral: α={best_mkd_coral['alpha']}  β={best_mkd_coral['beta']}  → QWK={best_mkd_coral['qwk']:.4f}")

  [checkpoint] Phase 'phase2a_mkd_avg' already done — loaded from /kaggle/working/phase_phase2a_mkd_avg.json
  PHASE 2a — Tune α for mkd_avg  (T=6.0) — already complete. Best: α=0.9  β=1.0  QWK=0.9030
  [checkpoint] Phase 'phase2b_camkd' already done — loaded from /kaggle/working/phase_phase2b_camkd.json
  PHASE 2b — Tune α for camkd  (T=6.0) — already complete. Best: α=0.5  β=1.0  QWK=0.9109
  [checkpoint] Phase 'phase2c_mkd_coral' already done — loaded from /kaggle/working/phase_phase2c_mkd_coral.json
  PHASE 2c — Tune α × β for mkd_coral  (T=6.0) — already complete. Best: α=0.9  β=0.5  QWK=0.9063

  PHASE 2 COMPLETE
  mkd_avg:   α=0.9                      → QWK=0.9030
  camkd:     α=0.5                        → QWK=0.9109
  mkd_coral: α=0.9  β=0.5  → QWK=0.9063


In [19]:
# ----------------------------------------------------------------------
# STEP 9 — final runs, 3 seeds each, using the cache again (fast)
# ----------------------------------------------------------------------
def evaluate_test_multikd(model, coral_student, loader):
    model.eval(); t_preds, t_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            if coral_student:
                feat   = model.classifier(model.features(imgs.to(device)).mean([2, 3]))
                logits = model.coral_head(feat)
                t_preds.extend(coral_predict(logits).cpu().numpy())
            else:
                t_preds.extend(model(imgs.to(device)).argmax(dim=1).cpu().numpy())
            t_labels.extend(labels.numpy())
    t_preds, t_labels = np.array(t_preds), np.array(t_labels)
    return {'qwk':  float(cohen_kappa_score(t_labels, t_preds, weights='quadratic')),
            'acc':  float(np.mean(t_labels == t_preds)),
            'mae':  float(np.mean(np.abs(t_labels - t_preds))),
            'rmse': float(np.sqrt(np.mean((t_labels - t_preds) ** 2)))}


def is_final_run_done(prefix, seed):
    p = f'{CKPT_DIR}/{prefix}_seed{seed}_resume.pth'
    if not os.path.exists(p): return False
    return torch.load(p, map_location='cpu', weights_only=False)['epoch'] >= NUM_EPOCHS - 1


def train_final_multikd(method, seed, alpha, temperature, beta=1.0):
    cfg         = MULTIKD_METHODS[method]
    prefix      = f'final_{method}'
    best_path   = f'{CKPT_DIR}/{prefix}_seed{seed}_best.pth'
    resume_path = f'{CKPT_DIR}/{prefix}_seed{seed}_resume.pth'

    if is_final_run_done(prefix, seed):
        print(f"  [checkpoint] {method} seed {seed} already complete — loading best weights")
        student = (make_mobilenet_coral(best_dropout) if cfg['coral_student']
                   else make_mobilenet_ce(best_dropout)).to(device)
        student.load_state_dict(torch.load(best_path, map_location=device,
                                           weights_only=False)['model_state_dict'])
        student.eval()
        return student

    set_seed(seed)
    loader  = make_train_loader_idx(best_bs)
    student = (make_mobilenet_coral(best_dropout) if cfg['coral_student']
               else make_mobilenet_ce(best_dropout)).to(device)
    opt     = optim.AdamW(student.parameters(), lr=best_lr, weight_decay=best_wd)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
    scaler  = torch.amp.GradScaler('cuda')
    ce_cache, coral_cache = TEACHER_CE_LOGITS.to(device), TEACHER_CORAL_LOGITS.to(device)
    history = {'val_qwk': []}
    best_qwk, start_epoch = -1.0, 0

    if os.path.exists(resume_path):
        start_epoch, history, best_qwk, _, _ = load_resume_ckpt(resume_path, student, opt, sched, scaler)

    for epoch in range(start_epoch, NUM_EPOCHS):
        student.train()
        for imgs, labels, idx in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t_ce_l, t_coral_l = ce_cache[idx], coral_cache[idx]
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                if cfg['coral_student']:
                    feat     = student.classifier(student.features(imgs).mean([2, 3]))
                    s_logits = student.coral_head(feat)
                else:
                    s_logits = student(imgs)
                loss = cfg['loss_fn'](s_logits, t_ce_l, t_coral_l, labels,
                                      alpha=alpha, temperature=temperature, beta=beta)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        student.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                if cfg['coral_student']:
                    feat   = student.classifier(student.features(imgs.to(device)).mean([2, 3]))
                    logits = student.coral_head(feat)
                    v_preds.extend(coral_predict(logits).cpu().numpy())
                else:
                    v_preds.extend(student(imgs.to(device)).argmax(dim=1).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        history['val_qwk'].append(qwk)
        print(f"  [{method} s{seed}] Epoch {epoch+1:03d} | QWK={qwk:.4f}")

        if qwk > best_qwk:
            best_qwk = qwk
            save_best_weights(best_path, epoch + 1, student, best_qwk)
        save_resume_ckpt(resume_path, epoch, student, opt, sched, scaler, history, best_qwk, [], [])

    student.load_state_dict(torch.load(best_path, map_location=device,
                                       weights_only=False)['model_state_dict'])
    student.eval()
    return student


final_configs = {
    'mkd_avg':   dict(alpha=best_mkd_avg['alpha'],   temperature=T_fixed, beta=1.0),
    'camkd':     dict(alpha=best_camkd['alpha'],     temperature=T_fixed, beta=1.0),
    'mkd_coral': dict(alpha=best_mkd_coral['alpha'], temperature=T_fixed, beta=best_mkd_coral['beta']),
}

final_test_results = {m: [] for m in MULTIKD_METHODS}
for method, params in final_configs.items():
    print(f"\n{'='*65}\n  FINAL RUNS — {method}  (params: {params})\n{'='*65}")
    for seed in [33, 81, 5]:
        student = train_final_multikd(method, seed, **params)
        tr = evaluate_test_multikd(student, MULTIKD_METHODS[method]['coral_student'], test_loader)
        tr['seed'] = seed
        final_test_results[method].append(tr)
        print(f"  [{method} s{seed}] TEST → QWK={tr['qwk']:.4f} | Acc={tr['acc']*100:.2f}% | "
              f"RMSE={tr['rmse']:.4f} | MAE={tr['mae']:.4f}")
        cleanup_memory()

print("\n" + "="*65)
print("  MULTI-TEACHER KD — FINAL TEST SUMMARY")
print("="*65)
for method, results in final_test_results.items():
    qwks, accs = [r['qwk'] for r in results], [r['acc']*100 for r in results]
    print(f"\n  {method}")
    for r in results:
        print(f"    seed {r['seed']:<3} QWK={r['qwk']:.4f}  Acc={r['acc']*100:.2f}%  "
              f"RMSE={r['rmse']:.4f}  MAE={r['mae']:.4f}")
    print(f"    mean  QWK={np.mean(qwks):.4f}±{np.std(qwks):.4f}  "
          f"Acc={np.mean(accs):.2f}%±{np.std(accs):.2f}%")


  FINAL RUNS — mkd_avg  (params: {'alpha': 0.9, 'temperature': 6.0, 'beta': 1.0})
  [checkpoint] mkd_avg seed 33 already complete — loading best weights
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 96.3MB/s]


  [mkd_avg s33] TEST → QWK=0.8391 | Acc=79.56% | RMSE=0.7119 | MAE=0.2888
  [checkpoint] mkd_avg seed 81 already complete — loading best weights
  [mkd_avg s81] TEST → QWK=0.8451 | Acc=80.11% | RMSE=0.7042 | MAE=0.2834
  [checkpoint] mkd_avg seed 5 already complete — loading best weights
  [mkd_avg s5] TEST → QWK=0.8588 | Acc=79.29% | RMSE=0.6664 | MAE=0.2752

  FINAL RUNS — camkd  (params: {'alpha': 0.5, 'temperature': 6.0, 'beta': 1.0})
  [checkpoint] camkd seed 33 already complete — loading best weights
  [camkd s33] TEST → QWK=0.8206 | Acc=77.38% | RMSE=0.7419 | MAE=0.3161
  [checkpoint] camkd seed 81 already complete — loading best weights
  [camkd s81] TEST → QWK=0.8478 | Acc=77.93% | RMSE=0.6905 | MAE=0.2916
  [checkpoint] camkd seed 5 already complete — loading best weights
  [camkd s5] TEST → QWK=0.8555 | Acc=81.20% | RMSE=0.6746 | MAE=0.2698

  FINAL RUNS — mkd_coral  (params: {'alpha': 0.9, 'temperature': 6.0, 'beta': 0.5})
  [checkpoint] mkd_coral seed 33 already complete —

In [20]:
# ========================================================================
# SINGLE-TEACHER KD — append these cells after STEP 9 in the notebook
#
# Reuses the existing `coral_kd_loss` (already defined, currently unused)
# and the existing TEACHER_CE_LOGITS cache from STEP 5 — no new teacher
# forward pass needed, same efficiency win as the multi-teacher phases.
#
# Design choice: `coral_kd_loss` expects a plain (B, NUM_CLASSES) softmax
# teacher (it does F.softmax(t_logits / T) directly), which matches the
# CE-ResNet50 teacher's logits, NOT the CORAL-EffNet-B3 teacher's (B, K-1)
# cumulative logits. So "single teacher" here = CE-ResNet50 teacher only,
# distilled into a CORAL student (mobilenet_coral). This is the natural
# single-teacher counterpart to mkd_coral in Phase 2c.
# ========================================================================

# ----------------------------------------------------------------------
# STEP 10 — single-teacher KD hyperparameter search
#           Phase 1: temperature (alpha, beta fixed)
#           Phase 2: alpha × beta (reusing best temperature)
# ----------------------------------------------------------------------
def train_search_single_kd(temperature, alpha=0.5, beta=1.0, num_epochs=SEARCH_EPOCHS):
    """Fast search trial: CE-ResNet50 teacher (cached logits) -> CORAL student."""
    loader = make_train_loader_idx(best_bs)
    set_seed(SEARCH_SEED)
    student = make_mobilenet_coral(best_dropout).to(device)
    opt     = optim.AdamW(student.parameters(), lr=best_lr, weight_decay=best_wd)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler  = torch.amp.GradScaler('cuda')
    ce_cache = TEACHER_CE_LOGITS.to(device)
    best_qwk = -1.0

    for epoch in range(num_epochs):
        student.train()
        for imgs, labels, idx in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t_ce_l = ce_cache[idx]   # cheap lookup, no teacher forward pass
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                feat     = student.classifier(student.features(imgs).mean([2, 3]))
                s_logits = student.coral_head(feat)
                loss     = coral_kd_loss(s_logits, t_ce_l, labels,
                                          alpha=alpha, temperature=temperature, beta=beta)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        student.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                feat   = student.classifier(student.features(imgs.to(device)).mean([2, 3]))
                logits = student.coral_head(feat)
                v_preds.extend(coral_predict(logits).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        if qwk > best_qwk: best_qwk = qwk

    del student; cleanup_memory()
    return best_qwk


def hypersearch_single_kd_temperature(phase_name='phase_s1_temperature'):
    results = load_phase_results(phase_name) or []
    done = {r['temperature'] for r in results}
    remaining = [t for t in T_SPACE if t not in done]
    print(f"\n{'='*65}\n  PHASE S1 — single-teacher (CE) KD, temperature search\n"
          f"  {len(T_SPACE)} trials total, {len(results)} already done, "
          f"{len(remaining)} remaining\n{'='*65}")

    for t in remaining:
        print(f"  Trial | T={t}", end='  ', flush=True)
        try:
            qwk = train_search_single_kd(temperature=t)   # alpha=0.5, beta=1.0 fixed
        except Exception as e:
            save_phase_results(phase_name, results)
            print(f"\n  [checkpoint] saved {len(results)}/{len(T_SPACE)} trials before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        results.append({'temperature': t, 'qwk': qwk})
        save_phase_results(phase_name, results)   # checkpoint after EVERY trial

    results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results(phase_name, results)
    return results


phase_s1_results = hypersearch_single_kd_temperature()
T_fixed_single = phase_s1_results[0]['temperature']
print(f"\n  PHASE S1 best: T={T_fixed_single}  QWK={phase_s1_results[0]['qwk']:.4f}")


def hypersearch_single_kd_alpha_beta(phase_name='phase_s2_alpha_beta'):
    trials = list(itertools.product(DISTILL_SEARCH['alpha'], BETA_SEARCH))
    results = load_phase_results(phase_name) or []
    done_keys = {(r['alpha'], r['beta']) for r in results}

    if len(results) == len(trials) and all((a, b) in done_keys for a, b in trials):
        results.sort(key=lambda x: x['qwk'], reverse=True)
        print(f"  PHASE S2 — already complete. Best: α={results[0]['alpha']}  "
              f"β={results[0]['beta']}  QWK={results[0]['qwk']:.4f}")
        return results

    remaining = [(a, b) for a, b in trials if (a, b) not in done_keys]
    print(f"\n{'='*65}\n  PHASE S2 — single-teacher (CE) KD, α × β search  "
          f"(T={T_fixed_single}, reused from Phase S1)\n"
          f"  {len(trials)} trials total, {len(results)} already done, "
          f"{len(remaining)} remaining\n{'='*65}")

    for alpha, beta in remaining:
        print(f"  Trial | α={alpha}  β={beta}", end='  ', flush=True)
        try:
            qwk = train_search_single_kd(temperature=T_fixed_single, alpha=alpha, beta=beta)
        except Exception as e:
            save_phase_results(phase_name, results)
            print(f"\n  [checkpoint] saved {len(results)}/{len(trials)} trials before error: {e}")
            raise
        print(f"→ QWK={qwk:.4f}")
        results.append({'alpha': alpha, 'beta': beta, 'qwk': qwk})
        save_phase_results(phase_name, results)   # checkpoint after EVERY trial

    results.sort(key=lambda x: x['qwk'], reverse=True)
    save_phase_results(phase_name, results)
    return results


phase_s2_results = hypersearch_single_kd_alpha_beta()
best_single_kd = phase_s2_results[0]
print(f"\n  PHASE S2 best: α={best_single_kd['alpha']}  β={best_single_kd['beta']}  "
      f"QWK={best_single_kd['qwk']:.4f}")

  [checkpoint] Phase 'phase_s1_temperature' already done — loaded from /kaggle/working/phase_phase_s1_temperature.json

  PHASE S1 — single-teacher (CE) KD, temperature search
  4 trials total, 4 already done, 0 remaining
  [checkpoint] Phase 'phase_s1_temperature' results saved → /kaggle/working/phase_phase_s1_temperature.json

  PHASE S1 best: T=6.0  QWK=0.9028
  [checkpoint] Phase 'phase_s2_alpha_beta' already done — loaded from /kaggle/working/phase_phase_s2_alpha_beta.json

  PHASE S2 — single-teacher (CE) KD, α × β search  (T=6.0, reused from Phase S1)
  12 trials total, 4 already done, 8 remaining
  Trial | α=0.5  β=1.0  → QWK=0.9028
  [checkpoint] Phase 'phase_s2_alpha_beta' results saved → /kaggle/working/phase_phase_s2_alpha_beta.json
  Trial | α=0.5  β=2.0  → QWK=0.8828
  [checkpoint] Phase 'phase_s2_alpha_beta' results saved → /kaggle/working/phase_phase_s2_alpha_beta.json
  Trial | α=0.7  β=0.5  → QWK=0.9023
  [checkpoint] Phase 'phase_s2_alpha_beta' results saved → /kaggl

In [21]:
# ----------------------------------------------------------------------
# STEP 11 — final single-teacher KD runs, 3 seeds, using the cache again
# ----------------------------------------------------------------------
def train_final_single_kd(seed, alpha, temperature, beta=1.0, num_epochs=NUM_EPOCHS):
    prefix      = 'final_single_kd'
    best_path   = f'{CKPT_DIR}/{prefix}_seed{seed}_best.pth'
    resume_path = f'{CKPT_DIR}/{prefix}_seed{seed}_resume.pth'

    if is_final_run_done(prefix, seed):
        print(f"  [checkpoint] single_kd seed {seed} already complete — loading best weights")
        student = make_mobilenet_coral(best_dropout).to(device)
        student.load_state_dict(torch.load(best_path, map_location=device,
                                           weights_only=False)['model_state_dict'])
        student.eval()
        return student

    set_seed(seed)
    loader  = make_train_loader_idx(best_bs)
    student = make_mobilenet_coral(best_dropout).to(device)
    opt     = optim.AdamW(student.parameters(), lr=best_lr, weight_decay=best_wd)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_epochs)
    scaler  = torch.amp.GradScaler('cuda')
    ce_cache = TEACHER_CE_LOGITS.to(device)
    history = {'val_qwk': []}
    best_qwk, start_epoch = -1.0, 0

    if os.path.exists(resume_path):
        start_epoch, history, best_qwk, _, _ = load_resume_ckpt(resume_path, student, opt, sched, scaler)

    for epoch in range(start_epoch, num_epochs):
        student.train()
        for imgs, labels, idx in loader:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            t_ce_l = ce_cache[idx]
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                feat     = student.classifier(student.features(imgs).mean([2, 3]))
                s_logits = student.coral_head(feat)
                loss     = coral_kd_loss(s_logits, t_ce_l, labels,
                                          alpha=alpha, temperature=temperature, beta=beta)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()

        student.eval(); v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                feat   = student.classifier(student.features(imgs.to(device)).mean([2, 3]))
                logits = student.coral_head(feat)
                v_preds.extend(coral_predict(logits).cpu().numpy())
                v_labels.extend(labels.numpy())
        qwk = cohen_kappa_score(np.array(v_labels), np.array(v_preds), weights='quadratic')
        history['val_qwk'].append(qwk)
        print(f"  [single_kd s{seed}] Epoch {epoch+1:03d} | QWK={qwk:.4f}")

        if qwk > best_qwk:
            best_qwk = qwk
            save_best_weights(best_path, epoch + 1, student, best_qwk)
        save_resume_ckpt(resume_path, epoch, student, opt, sched, scaler, history, best_qwk, [], [])

    student.load_state_dict(torch.load(best_path, map_location=device,
                                       weights_only=False)['model_state_dict'])
    student.eval()
    return student


single_kd_config = dict(alpha=best_single_kd['alpha'],
                         temperature=T_fixed_single,
                         beta=best_single_kd['beta'])

final_test_results['single_kd'] = []
print(f"\n{'='*65}\n  FINAL RUNS — single_kd  (params: {single_kd_config})\n{'='*65}")
for seed in [33, 81, 5]:
    student = train_final_single_kd(seed, **single_kd_config)
    tr = evaluate_test_multikd(student, coral_student=True, loader=test_loader)
    tr['seed'] = seed
    final_test_results['single_kd'].append(tr)
    print(f"  [single_kd s{seed}] TEST → QWK={tr['qwk']:.4f} | Acc={tr['acc']*100:.2f}% | "
          f"RMSE={tr['rmse']:.4f} | MAE={tr['mae']:.4f}")
    cleanup_memory()


  FINAL RUNS — single_kd  (params: {'alpha': 0.9, 'temperature': 6.0, 'beta': 1.0})
  [single_kd s33] Epoch 001 | QWK=0.7246
  [single_kd s33] Epoch 002 | QWK=0.7208
  [single_kd s33] Epoch 003 | QWK=0.7079
  [single_kd s33] Epoch 004 | QWK=0.7192
  [single_kd s33] Epoch 005 | QWK=0.7376
  [single_kd s33] Epoch 006 | QWK=0.7918
  [single_kd s33] Epoch 007 | QWK=0.7500
  [single_kd s33] Epoch 008 | QWK=0.7916
  [single_kd s33] Epoch 009 | QWK=0.8680
  [single_kd s33] Epoch 010 | QWK=0.8560
  [single_kd s33] Epoch 011 | QWK=0.8776
  [single_kd s33] Epoch 012 | QWK=0.8436
  [single_kd s33] Epoch 013 | QWK=0.8976
  [single_kd s33] Epoch 014 | QWK=0.8826
  [single_kd s33] Epoch 015 | QWK=0.8757
  [single_kd s33] Epoch 016 | QWK=0.8843
  [single_kd s33] Epoch 017 | QWK=0.8675
  [single_kd s33] Epoch 018 | QWK=0.8731
  [single_kd s33] Epoch 019 | QWK=0.8715
  [single_kd s33] Epoch 020 | QWK=0.8783
  [single_kd s33] Epoch 021 | QWK=0.8805
  [single_kd s33] Epoch 022 | QWK=0.8721
  [single_kd 

In [22]:
# ----------------------------------------------------------------------
# STEP 12 — updated summary, single-teacher vs multi-teacher KD
# ----------------------------------------------------------------------
print("\n" + "="*65)
print("  SINGLE- vs MULTI-TEACHER KD — FINAL TEST SUMMARY")
print("="*65)
for method, results in final_test_results.items():
    qwks, accs = [r['qwk'] for r in results], [r['acc']*100 for r in results]
    print(f"\n  {method}")
    for r in results:
        print(f"    seed {r['seed']:<3} QWK={r['qwk']:.4f}  Acc={r['acc']*100:.2f}%  "
              f"RMSE={r['rmse']:.4f}  MAE={r['mae']:.4f}")
    print(f"    mean  QWK={np.mean(qwks):.4f}±{np.std(qwks):.4f}  "
          f"Acc={np.mean(accs):.2f}%±{np.std(accs):.2f}%")


  SINGLE- vs MULTI-TEACHER KD — FINAL TEST SUMMARY

  mkd_avg
    seed 33  QWK=0.8391  Acc=79.56%  RMSE=0.7119  MAE=0.2888
    seed 81  QWK=0.8451  Acc=80.11%  RMSE=0.7042  MAE=0.2834
    seed 5   QWK=0.8588  Acc=79.29%  RMSE=0.6664  MAE=0.2752
    mean  QWK=0.8477±0.0082  Acc=79.65%±0.34%

  camkd
    seed 33  QWK=0.8206  Acc=77.38%  RMSE=0.7419  MAE=0.3161
    seed 81  QWK=0.8478  Acc=77.93%  RMSE=0.6905  MAE=0.2916
    seed 5   QWK=0.8555  Acc=81.20%  RMSE=0.6746  MAE=0.2698
    mean  QWK=0.8413±0.0150  Acc=78.84%±1.68%

  mkd_coral
    seed 33  QWK=0.8620  Acc=80.11%  RMSE=0.6478  MAE=0.2616
    seed 81  QWK=0.8727  Acc=77.11%  RMSE=0.6350  MAE=0.2834
    seed 5   QWK=0.8732  Acc=79.84%  RMSE=0.6242  MAE=0.2589
    mean  QWK=0.8693±0.0052  Acc=79.02%±1.35%

  single_kd
    seed 33  QWK=0.8366  Acc=74.66%  RMSE=0.7456  MAE=0.3379
    seed 81  QWK=0.8502  Acc=76.02%  RMSE=0.6766  MAE=0.2997
    seed 5   QWK=0.8653  Acc=81.74%  RMSE=0.6393  MAE=0.2507
    mean  QWK=0.8507±0.0117  Acc

In [2]:
!pip -q install -U huggingface_hub

from kaggle_secrets import UserSecretsClient
from huggingface_hub import upload_folder
from huggingface_hub import HfApi, create_repo, upload_folder

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

create_repo(
    repo_id="almirari/hyperparameter-search-rev",
    token=HF_TOKEN,
    exist_ok=True,
)


upload_folder(
    repo_id="almirari/hyperparameter-search-rev",
    folder_path="/kaggle/working",
    token=HF_TOKEN
)

CommitInfo(commit_url='https://huggingface.co/almirari/hyperparameter-search-rev/commit/a4bf7a64dd4a3b60dc578e48bd0e1a144c037f7e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='a4bf7a64dd4a3b60dc578e48bd0e1a144c037f7e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/almirari/hyperparameter-search-rev', endpoint='https://huggingface.co', repo_type='model', repo_id='almirari/hyperparameter-search-rev'), pr_revision=None, pr_num=None)